In [2]:
import pandas as pd
import numpy as np

meta = pd.read_csv("../data/processed/response_meta.csv", index_col=0)
drugs = sorted(meta["product_name"].unique())
print(len(drugs))   # 188

def clean(name):
    name = name.split(" (")[0]
    for salt in ["2HCl", "HCl", "Mesylate", "Sodium", "Dihydrochloride"]:
        name = name.replace(salt, "")
    return name.strip()

cleaned = {d: clean(d) for d in drugs}
c = pd.Series(cleaned)

print("drugs:", len(drugs))
print("unique cleaned names:", c.nunique())
print("empty names:", (c == "").sum())
print("duplicates:", c[c.duplicated(keep=False)].to_dict())

188
drugs: 188
unique cleaned names: 188
empty names: 0
duplicates: {}


In [3]:
import pubchempy as pcp

smiles = {}
for d in drugs:
    try:
        res = pcp.get_compounds(clean(d), "name")
        smiles[d] = res[0].isomeric_smiles if res else None
    except Exception:
        smiles[d] = None

sm = pd.Series(smiles, name="smiles")
print("missing:", sm.isna().sum())
print(sm[sm.isna()].index.tolist())

sm.to_csv("../data/external/drug_smiles.csv")

/tmp/ipykernel_12896/1573017319.py:7: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  smiles[d] = res[0].isomeric_smiles if res else None


missing: 4
['Fedratinib (SAR302503, TG101348)', 'Glesatinib?(MGCD265)', 'Pracinostat (SB939)', 'Valproic acid sodium salt (Sodium valproate)']


In [4]:
#Fix the 4 missing
import pubchempy as pcp

manual = {
    "Fedratinib (SAR302503, TG101348)": "Fedratinib",
    "Glesatinib?(MGCD265)": "Glesatinib",
    "Pracinostat (SB939)": "Pracinostat",
    "Valproic acid sodium salt (Sodium valproate)": "Valproic acid",
}

for d, name in manual.items():
    res = pcp.get_compounds(name, "name")
    sm[d] = res[0].isomeric_smiles if res else None
    print(d, "->", "OK" if sm[d] else "still missing")

print("missing now:", sm.isna().sum())
sm.to_csv("../data/external/drug_smiles.csv")

/tmp/ipykernel_12896/297462114.py:13: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  sm[d] = res[0].isomeric_smiles if res else None


Fedratinib (SAR302503, TG101348) -> OK
Glesatinib?(MGCD265) -> OK
Pracinostat (SB939) -> OK
Valproic acid sodium salt (Sodium valproate) -> OK
missing now: 0


In [5]:
#Make fingerprints and count unique ones
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

fps = {}
for d, s in sm.items():
    s = max(s.split("."), key=len)      # remove salt, keep largest piece
    mol = Chem.MolFromSmiles(s)
    if mol is None:
        print("could not read:", d)
        continue
    fps[d] = gen.GetFingerprintAsNumPy(mol)

fp = pd.DataFrame(fps).T
print("drugs with fingerprint:", len(fp))
print("unique fingerprints:", len(fp.drop_duplicates()))

drugs with fingerprint: 188
unique fingerprints: 186


In [6]:
dup = fp[fp.duplicated(keep=False)]
print(dup.index.tolist())

['ENMD-2076', 'ENMD-2076 L-(+)-Tartaric acid ', 'Ruxolitinib (INCB018424)', 'S-Ruxolitinib (INCB018424)']


In [7]:
# one group ID per unique fingerprint
fp_key = fp.astype(str).agg("".join, axis=1)
group_id = pd.Series(pd.factorize(fp_key)[0], index=fp.index, name="group")

print("drugs:", len(group_id))
print("groups:", group_id.nunique())   # should be 186
print(group_id[["ENMD-2076", "ENMD-2076 L-(+)-Tartaric acid "]])  # same number

drugs: 188
groups: 186
ENMD-2076                         60
ENMD-2076 L-(+)-Tartaric acid     60
Name: group, dtype: int64


In [8]:
fp.to_csv("../data/processed/drug_fingerprints.csv")
group_id.to_csv("../data/processed/drug_groups.csv")

In [9]:
#Build X

delta = pd.read_csv("../data/processed/response_delta.csv", index_col=0)
meta = pd.read_csv("../data/processed/response_meta.csv", index_col=0)

X = fp.loc[meta["product_name"]].values          # fingerprint per row
X = pd.DataFrame(X, index=meta.index, columns=[f"bit{i}" for i in range(1024)])
X["log_dose"] = np.log10(meta["dose"])

print(X.shape)   # (750, 1025)

(750, 1025)


In [10]:
#Drop bits that never change
keep = X.nunique() > 1
X = X.loc[:, keep]
print("columns left:", X.shape[1])

columns left: 970


In [11]:
#Check row order and add groups
print("same order:", (X.index == delta.index).all())   # must be True

row_groups = group_id.loc[meta["product_name"]].values
print("groups:", len(set(row_groups)))   # should be 186

same order: True
groups: 186


In [12]:
#4. Tanimoto sanity check
def tanimoto(a, b):
    return (a & b).sum() / (a | b).sum()

hdac = [d for d in fp.index if "inostat" in d]
other = [d for d in fp.index if "inostat" not in d]
print("HDAC drugs:", hdac)

v = fp.values.astype(bool)
pos = {d: i for i, d in enumerate(fp.index)}

within = [tanimoto(v[pos[a]], v[pos[b]]) for a in hdac for b in hdac if a < b]
between = [tanimoto(v[pos[a]], v[pos[b]]) for a in hdac for b in other]

print("HDAC vs HDAC:", round(np.mean(within), 3))
print("HDAC vs others:", round(np.mean(between), 3))

HDAC drugs: ['Abexinostat (PCI-24781)', 'Belinostat (PXD101)', 'Dacinostat (LAQ824)', 'Droxinostat', 'Entinostat (MS-275)', 'Givinostat (ITF2357)', 'Mocetinostat (MGCD0103)', 'Panobinostat (LBH589)', 'Pracinostat (SB939)', 'Quisinostat (JNJ-26481585) 2HCl', 'Resminostat', 'Tucidinostat (Chidamide)']
HDAC vs HDAC: 0.219
HDAC vs others: 0.129


In [13]:
X.to_csv("../data/processed/X_features.csv")
pd.Series(row_groups, index=X.index, name="group").to_csv("../data/processed/row_groups.csv")

In [14]:
X = pd.read_csv("../data/processed/X_features.csv", index_col=0)
Y = pd.read_csv("../data/processed/response_delta.csv", index_col=0)

print("X:", X.shape)
print("Y:", Y.shape)
print("same rows:", (X.index == Y.index).all())

X: (750, 970)
Y: (750, 2000)
same rows: True


In [15]:
rows = [r for r in X.index if r.startswith("Panobinostat")]

print("X (input):")
print(X.loc[rows, list(X.columns[:5]) + ["log_dose"]])

print("Y (output):")
print(Y.loc[rows, Y.columns[:5]].round(3))

X (input):
                             bit1  bit2  bit3  bit4  bit5  log_dose
product_dose                                                       
Panobinostat (LBH589)_1000      0     0     0     0     0       3.0
Panobinostat (LBH589)_10000     0     0     0     0     0       4.0
Panobinostat (LBH589)_100       0     0     0     0     0       2.0
Panobinostat (LBH589)_10        0     0     0     0     0       1.0
Y (output):
                              GCLC   M6PR  KLHL13  CYP26B1   TFPI
Panobinostat (LBH589)_1000   0.352  0.141   0.781    0.000 -0.399
Panobinostat (LBH589)_10000  0.287  0.132   0.426    0.007 -0.396
Panobinostat (LBH589)_100    0.438 -0.004   0.211   -0.004 -0.364
Panobinostat (LBH589)_10    -0.415  0.080  -0.000    0.031  0.020
